In [4]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

In [10]:
MODEL = "gemini-2.5-flash-lite"

In [11]:
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

In [12]:
pregunta = "¿En que año llego el ser humano a la luna por primera vez?"

In [13]:
print("Pregunta: ", pregunta)

Pregunta:  ¿En que año llego el ser humano a la luna por primera vez?


# INVOCACION

In [14]:
respuesta = llm.invoke(pregunta)

In [15]:
print("Respuesta del modelo: ", respuesta.content)

Respuesta del modelo:  El ser humano llegó a la Luna por primera vez en **1969**.


# Stream

In [16]:
for chunk in llm.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

Parrots have colorful feathers for| a variety of fascinating reasons, and it's a combination of **survival, communication|, and reproduction**. Here's a breakdown of the key factors:

**1. Camouflage:**

*   **Blending In:** While many people associate parrots with bright, conspicuous colors, not all of them are. Many species have green| and brown plumage, which helps them blend seamlessly into the leafy canopy of their rainforest habitats. This is crucial for avoiding predators like raptors and snakes.
*   **Species-Specific Camouflage:** The specific colors and patterns can also help| them blend into particular environments or even specific types of trees.

**2. Communication and Social Signaling:**

*   **Species Recognition:** The vibrant and unique color patterns of a parrot's plumage act like a "uniform" for their| species. This helps them recognize their own kind from a distance, which is important for flocking, mating, and avoiding interbreeding with other species.
*   **Indivi

# Batch

In [17]:
responses = llm.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?"
])

for response in responses:
  print(response)

content="Parrots have colorful feathers for a variety of reasons, and it's a fascinating interplay of **evolution, communication, and survival**. Here's a breakdown of the main factors:\n\n**1. Mating and Sexual Selection:**\n\n* **Attracting a Mate:** This is arguably the most significant reason for vibrant coloration in many parrot species. Brighter, more elaborate plumage can signal a parrot's health, genetic quality, and overall fitness to potential mates. A visually striking bird is more likely to catch the eye of a desirable partner.\n* **Species Recognition:** In areas where multiple parrot species live together, distinct color patterns help individuals recognize members of their own species. This prevents hybridization and ensures that they mate with appropriate partners.\n* **Courtship Displays:** The colors are often used in elaborate courtship rituals. Parrots may puff up their feathers, display specific patches of color, or perform dances to showcase their plumage.\n\n**2. 

# Herramientas

In [18]:
!pip install langchain

In [25]:
from langchain.tools import tool

In [26]:
@tool
def get_weather(location: str) -> str:
  """Get the weather at a location"""
  return f"It is sunny in {location}"

In [21]:
model_with_tools = llm.bind_tools([get_weather])

In [22]:
response = model_with_tools.invoke("What is the weather in Boston?")

In [23]:
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019db110-bb88-7611-9e99-fe56735d1c85-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '30b0fdcb-a215-4e2f-a312-7584ab8c1206', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 15, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}})

In [24]:
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


# Templates

In [27]:
!pip install langchain-core

In [28]:
from langchain_core.prompts import PromptTemplate

In [30]:
template = """
Actua como un experto en {domain}.

Explica el siguiente concepto:
{concept}

A un nivel {nivel}

Usa ejemplos.
"""

In [31]:
promt = PromptTemplate.from_template(template)

In [32]:
chain = promt | llm

In [35]:
reponse = chain.invoke({
    "domain": "machine learning",
    "concept": "overfitting",
    "nivel": "principiante"
})

In [37]:
reponse.content

'¡Absolutamente! Ponte cómodo, porque vamos a desentrañar el concepto de "overfitting" (sobreajuste) de una manera súper sencilla, como si estuviéramos hablando de aprender algo nuevo.\n\nImagina que eres un estudiante y te están preparando para un examen de historia sobre la Segunda Guerra Mundial.\n\n**¿Qué es el Overfitting?**\n\nEl overfitting ocurre cuando un modelo de machine learning se vuelve **demasiado bueno** aprendiendo los datos con los que se entrena, hasta el punto de que **memoriza los detalles específicos y hasta los "ruidos" o errores** de esos datos, en lugar de aprender los patrones generales y las reglas subyacentes.\n\nPiensa en ello como memorizar las respuestas exactas de un examen de práctica, en lugar de entender los conceptos.\n\n**Ejemplo Sencillo: El Examen de Historia**\n\nSupongamos que tu profesor te da un conjunto de preguntas y respuestas para que estudies para el examen de historia sobre la Segunda Guerra Mundial.\n\n*   **Entrenamiento:** Tienes un l

# Anatomía de un Prompt

In [39]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [38]:
human_basico = "Analiza este texto"

human_completo = """
TAREA: Analiza el sentimiento del siguiente texto de reseña de producto.

TEXTO A ANALIZAR:
"El producto llegó puntual pero la calidad es mediocre para el precio que cobran.
Esperaba mucho más basándome en las fotos de la web. El servicio de atención al
cliente fue amable cuando reclamé, así que algo es algo."

INSTRUCCIONES:
- Identifica el sentimiento general (positivo/negativo/mixto)
- Lista los aspectos positivos mencionados
- Lista los aspectos negativos mencionados
- Proporciona una puntuación del 1 al 5

FORMATO DE SALIDA: Responde en formato estructurado con secciones claramente separadas.
"""

In [41]:
print(llm.invoke(human_completo).content)

## Análisis de Sentimiento de Reseña de Producto

**Sentimiento General:** Mixto

**Aspectos Positivos Mencionados:**
*   El producto llegó puntual.
*   El servicio de atención al cliente fue amable.

**Aspectos Negativos Mencionados:**
*   La calidad es mediocre para el precio.
*   Las expectativas basadas en las fotos de la web no se cumplieron.

**Puntuación (1-5):** 2/5


In [52]:
messages_con_prefijo = [
    SystemMessage(content="Eres un extractor de información. Siempre respondes en JSON válido."),
    HumanMessage(content="Extrae la información de este texto: 'Ana García, 32 años, vive en Madrid, trabaja como ingeniera de software en Telefónica desde 2019.'"),
    AIMessage(content='{"nombre":')  # Prefijo que fuerza el formato
]

In [53]:
resp = llm.invoke(messages_con_prefijo)

In [55]:
print(resp.content)

```json
{
  "nombre": "Ana García",
  "edad": 32,
  "ciudad": "Madrid",
  "profesion": "ingeniera de software",
  "empresa": "Telefónica",
  "año_inicio_trabajo": 2019
}
```


In [56]:
!pip install pydantic

In [57]:
## JsonOutputParser

In [75]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [63]:
parser_json = JsonOutputParser()

In [64]:
template_json = ChatPromptTemplate.from_messages([
    ("system", "Siempre responde con un JSON válido"),
    ("human", "{pregunta}")
])

In [65]:
cadena_json = template_json | llm | parser_json

In [66]:
resultado = cadena_json.invoke({
    "pregunta": "Dame informacion sobre Madrid: poblacion, pais, ubicacion, comida tipica"
})

In [67]:
import json

In [68]:
print(f"Tipo resultado: {type(resultado).__name__}")
print("\nContenido:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

Tipo resultado: dict

Contenido:
{
  "ciudad": "Madrid",
  "pais": "España",
  "ubicacion": {
    "coordenadas": "40°25′N 3°42′O",
    "comunidad_autonoma": "Comunidad de Madrid",
    "provincia": "Madrid",
    "region_geografica": "Meseta Central"
  },
  "poblacion": {
    "total_municipio": 3305406,
    "fecha_estimacion": "2023",
    "densidad_municipio": "5.300 hab/km²",
    "area_metropolitana": 6779847,
    "fecha_estimacion_area_metropolitana": "2023"
  },
  "comida_tipica": [
    {
      "nombre": "Cocido madrileño",
      "descripcion": "Un guiso contundente de garbanzos, verduras y carnes variadas (morcillo, gallina, chorizo, morcilla, tocino)."
    },
    {
      "nombre": "Bocadillo de calamares",
      "descripcion": "Un clásico de la gastronomía madrileña. Bocadillo relleno de calamares fritos y rebozados, a menudo con mayonesa o alioli."
    },
    {
      "nombre": "Churros con chocolate",
      "descripcion": "Masa frita en forma de bastones, servida con una taza de ch

In [85]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional

In [86]:
class AnalisisSentimiento(BaseModel):
  clasificacion: str = Field(
      description="Clasificacion del sentimiento:"
  )
  puntuacion: int = Field(
      description="Puntuacion del 1 al 5, donde 1 es malo y 5 es bueno"
  )
  aspectos_negativos: List[str] = Field(
      description="Lista de aspectos negativos del texto"
  )
  resumen: str = Field(
      descripcion="Resumen breve de la reseña"
  )
  recomendaria: bool = Field(
      description="Si el cliente recomendaria o no"
  )

  @field_validator('clasificacion')
  @classmethod
  def validar_clasificacion(cls, v):
      permitidos = {'POSITIVO', 'NEGATIVO', 'MIXTO'}
      if v.upper() not in permitidos:
          raise ValueError(f"Clasificación debe ser una de: {permitidos}")
      return v.upper()

/tmp/ipykernel_1750/609742867.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descripcion'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  resumen: str = Field(


In [87]:
parser_pydantic = PydanticOutputParser(pydantic_object=AnalisisSentimiento)

In [88]:
template_pydantic = ChatPromptTemplate.from_messages([
    ("system", """Eres un analizador de reseñas de productos.

{format_instructions}"""),
    ("human", "Analiza esta reseña:\n\n{resena}")
])

In [89]:
cadena_pydantic = template_pydantic | llm | parser_pydantic

In [90]:
RESENA_COMPLEJA = """
Compré este robot de cocina hace 3 meses y tengo sentimientos encontrados.
La potencia es brutal y pica verduras perfectamente, pero el ruido es ensordecedor.
El material parece de calidad y el diseño es elegante. Sin embargo, la app es un desastre
absoluto, se cuelga constantemente. El servicio técnico fue rapidísimo cuando reporté el problema.
Por el precio (450€) esperaba más, especialmente en software. El hardware, un 9; el software, un 3.
"""

In [92]:
resultado = cadena_pydantic.invoke({
    "resena": RESENA_COMPLEJA,
    "format_instructions": parser_pydantic.get_format_instructions()
    })

print(f"Tipo resultado: {type(resultado).__name__}")
print(f"\nClasificación: {resultado.clasificacion}")
print(f"Puntuación: {resultado.puntuacion}/5")
print(f"¿Recomendaría?: {'Sí' if resultado.recomendaria else 'No'}")
print(f"\nAspectos negativos:")
for asp in resultado.aspectos_negativos:
    print(f"  ❌ {asp}")
print(f"\nResumen: {resultado.resumen}")

Tipo resultado: AnalisisSentimiento

Clasificación: MIXTO
Puntuación: 3/5
¿Recomendaría?: No

Aspectos positivos:

Aspectos negativos:
  ❌ Ruido ensordecedor
  ❌ La app es un desastre y se cuelga constantemente
  ❌ Esperaba más por el precio, especialmente en software

Resumen: El robot de cocina tiene una potencia y materiales excelentes, pero sufre de un ruido muy alto y una aplicación móvil muy inestable. El servicio técnico respondió rápidamente.


In [97]:
resultado

AnalisisSentimiento(clasificacion='MIXTO', puntuacion=3, aspectos_negativos=['Ruido ensordecedor', 'La app es un desastre y se cuelga constantemente', 'Esperaba más por el precio, especialmente en software'], resumen='El robot de cocina tiene una potencia y materiales excelentes, pero sufre de un ruido muy alto y una aplicación móvil muy inestable. El servicio técnico respondió rápidamente.', recomendaria=False)